## 데이터 로드

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


데이터 로드

In [7]:
import pandas as pd

train = pd.read_csv(
    "/content/drive/MyDrive/movie/ua.base",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

test = pd.read_csv(
    "/content/drive/MyDrive/movie/ua.test",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

train.head()

,user_id,movie_id,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,3,4,878542960
3,1,4,3,876893119
4,1,5,3,889751712


timestamp 제거

In [8]:
train = train.drop("timestamp", axis=1)
test = test.drop("timestamp", axis=1)

print(train.shape)
print(test.shape)

print(train.head())
print(test.head())

(90570, 3)
(9430, 3)
   user_id  movie_id  rating
0        1         1       5
1        1         2       3
2        1         3       4
3        1         4       3
4        1         5       3
   user_id  movie_id  rating
0        1        20       4
1        1        33       4
2        1        61       4
3        1       117       3
4        1       155       2


ranking matrix 생성

In [9]:
train_matrix = train.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

train_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## train/val data 분리

In [10]:
from sklearn.model_selection import train_test_split

train_inner, val = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["user_id"]
)

print(train_inner.shape)
print(val.shape)

(72456, 3)
(18114, 3)


## Item-Based CF - zero fill

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

User-Item Matrix 생성

In [12]:
train_matrix = train_inner.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

train_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1671,1672,1674,1675,1676,1677,1678,1679,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,NaN,NaN,4.0,NaN,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


결측값 0으로 채우기

In [13]:
train_matrix_filled = train_matrix.fillna(0)

Item-Item Cosine Similarity 계산

In [14]:
item_similarity = cosine_similarity(train_matrix_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=train_matrix.columns,
    columns=train_matrix.columns
)

item_similarity_df.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1671,1672,1674,1675,1676,1677,1678,1679,1681,1682
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.314152,0.246249,0.357051,0.209860,0.083805,0.455583,0.345120,0.347344,0.268149,...,0.0,0.039559,0.0,0.000000,0.000000,0.041959,0.0,0.0,0.055945,0.000000
2,0.314152,1.000000,0.169440,0.369968,0.241500,0.026413,0.278978,0.238052,0.200265,0.114372,...,0.0,0.067015,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.094774,0.094774
3,0.246249,0.169440,1.000000,0.254598,0.134046,0.075822,0.280278,0.141883,0.205756,0.141933,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
4,0.357051,0.369968,0.254598,1.000000,0.218684,0.063104,0.432511,0.348277,0.357961,0.233410,...,0.0,0.044682,0.0,0.105316,0.105316,0.042126,0.0,0.0,0.063189,0.084253
5,0.209860,0.241500,0.134046,0.218684,1.000000,0.019818,0.248801,0.185328,0.163054,0.063803,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.106668


Item-Based CF 예측 함수

In [15]:
def predict_item_based(user_id, movie_id, train_matrix, item_similarity_df):
    # user가 train에 없는 경우
    if user_id not in train_matrix.index:
        return train_matrix.stack().mean()

    # movie가 train similarity에 없는 경우
    if movie_id not in item_similarity_df.index:
        return train_matrix.loc[user_id].mean()

    # 해당 사용자의 평점 벡터
    user_ratings = train_matrix.loc[user_id].dropna()

    # 사용자가 아무 영화도 평가하지 않은 경우
    if len(user_ratings) == 0:
        return train_matrix.stack().mean()

    # 사용자가 평가한 영화들과 target movie의 유사도
    similarities = item_similarity_df.loc[movie_id, user_ratings.index]

    # 가중합 계산
    numerator = np.dot(similarities, user_ratings)
    denominator = np.sum(np.abs(similarities))

    if denominator == 0:
        return user_ratings.mean()

    pred = numerator / denominator

    # 평점 범위 제한
    pred = np.clip(pred, 1, 5)

    return pred

Validation 데이터 예측

In [16]:
val_preds = []

for row in val.itertuples():
    pred = predict_item_based(
        row.user_id,
        row.movie_id,
        train_matrix,
        item_similarity_df
    )
    val_preds.append(pred)

val["item_cf_pred"] = val_preds
val.head()

,user_id,movie_id,rating,item_cf_pred
30016,311,89,5,3.922228
51012,496,485,3,2.998037
79374,826,678,4,3.884997
30415,312,611,5,4.448797
69284,699,95,3,3.103516


RMSE 계산



In [17]:
rmse_basic = np.sqrt(
    mean_squared_error(val["rating"], val["item_cf_pred"])
)

print("Basic Item-Based CF Validation RMSE:", rmse_basic)

Basic Item-Based CF Validation RMSE: 1.0105797462291823


## Item-Based CF - Mean-Centered

mean centering

In [18]:
user_means = train_matrix.mean(axis=1)

In [19]:
train_centered = train_matrix.sub(user_means, axis=0)
train_centered_filled = train_centered.fillna(0)

아이템 유사도 계산 - 0 으로 채운 값으로

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity_centered = cosine_similarity(train_centered_filled.T)

item_similarity_centered_df = pd.DataFrame(
    item_similarity_centered,
    index=train_centered.columns,
    columns=train_centered.columns
)

예측 함수

In [21]:
import numpy as np

global_mean = train_matrix.stack().mean()

def predict_item_based_centered(user_id, movie_id, train_matrix, item_similarity_df, user_means):
    # cold-start user
    if user_id not in train_matrix.index:
        return global_mean

    # 기준 사용자 평균
    user_mean = user_means.loc[user_id]

    # cold-start item
    if movie_id not in item_similarity_df.index:
        return np.clip(user_mean, 1, 5)

    # 사용자가 평가한 영화들
    user_ratings = train_matrix.loc[user_id].dropna()

    if len(user_ratings) == 0:
        return global_mean

    # mean-centered rating
    centered_ratings = user_ratings - user_mean

    # target item과 사용자가 평가한 item들의 유사도
    similarities = item_similarity_df.loc[movie_id, user_ratings.index]

    numerator = np.dot(similarities, centered_ratings)
    denominator = np.sum(np.abs(similarities))

    if denominator == 0:
        return np.clip(user_mean, 1, 5)

    pred = user_mean + numerator / denominator

    return np.clip(pred, 1, 5)

validation 예측

In [22]:
val_centered_preds = []

for row in val.itertuples():
    pred = predict_item_based_centered(
        row.user_id,
        row.movie_id,
        train_matrix,
        item_similarity_centered_df,
        user_means
    )
    val_centered_preds.append(pred)

val["item_cf_centered_pred"] = val_centered_preds

RMSE 계산

In [23]:
from sklearn.metrics import mean_squared_error

rmse_centered = np.sqrt(
    mean_squared_error(val["rating"], val["item_cf_centered_pred"])
)

print("Mean-Centered Item-Based CF Validation RMSE:", rmse_centered)

Mean-Centered Item-Based CF Validation RMSE: 0.9473802020437072


RMSE 비교

In [24]:
print("Basic Item-Based CF RMSE:", rmse_basic)
print("Mean-Centered Item-Based CF RMSE:", rmse_centered)

Basic Item-Based CF RMSE: 1.0105797462291823
Mean-Centered Item-Based CF RMSE: 0.9473802020437072


## Item-Based CF - Mean-Centered + top-k

예측 함수

In [25]:
global_mean = train_matrix.stack().mean()

In [26]:
def predict_item_based_centered_topk(
    user_id,
    movie_id,
    train_matrix,
    item_similarity_df,
    user_means,
    K=30
):
    # cold-start user
    if user_id not in train_matrix.index:
        return global_mean

    user_mean = user_means.loc[user_id]

    # cold-start item
    if movie_id not in item_similarity_df.index:
        return np.clip(user_mean, 1, 5)

    # 사용자가 평가한 영화들
    user_ratings = train_matrix.loc[user_id].dropna()

    if len(user_ratings) == 0:
        return global_mean

    # target movie와 사용자가 평가한 영화들 간 유사도
    similarities = item_similarity_df.loc[movie_id, user_ratings.index]

    # 자기 자신 제외
    similarities = similarities.drop(labels=[movie_id], errors="ignore")
    user_ratings = user_ratings.loc[similarities.index]

    # 유사도 높은 Top-K만 선택
    top_k_similarities = similarities.sort_values(
        ascending=False
    ).head(K)

    top_k_ratings = user_ratings.loc[top_k_similarities.index]

    if len(top_k_similarities) == 0:
        return np.clip(user_mean, 1, 5)

    # mean-centered rating
    centered_ratings = top_k_ratings - user_mean

    numerator = np.dot(top_k_similarities, centered_ratings)
    denominator = np.sum(np.abs(top_k_similarities))

    if denominator == 0:
        return np.clip(user_mean, 1, 5)

    pred = user_mean + numerator / denominator

    return np.clip(pred, 1, 5)

Top-K 별 RMSE 비교

In [27]:
K_list = [5, 10, 20, 30, 40, 50, 100]

results = []

for K in K_list:
    preds = []

    for row in val.itertuples():
        pred = predict_item_based_centered_topk(
            row.user_id,
            row.movie_id,
            train_matrix,
            item_similarity_centered_df,
            user_means,
            K=K
        )
        preds.append(pred)

    rmse = np.sqrt(mean_squared_error(val["rating"], preds))

    results.append({
        "model": f"Mean-Centered Item-CF Top-{K}",
        "K": K,
        "RMSE": rmse
    })

results_df = pd.DataFrame(results)
results_df

,model,K,RMSE
0,Mean-Centered Item-CF Top-5,5,0.972080
1,Mean-Centered Item-CF Top-10,10,0.940072
2,Mean-Centered Item-CF Top-20,20,0.931331
3,Mean-Centered Item-CF Top-30,30,0.930488
4,Mean-Centered Item-CF Top-40,40,0.932327
5,Mean-Centered Item-CF Top-50,50,0.934035
6,Mean-Centered Item-CF Top-100,100,0.940788


In [28]:
# Basic Item-CF RMSE
rmse_basic = np.sqrt(
    mean_squared_error(val["rating"], val["item_cf_pred"])
)

# Mean-Centered Item-CF RMSE
rmse_centered = np.sqrt(
    mean_squared_error(val["rating"], val["item_cf_centered_pred"])
)

# Top-K 중 최고 성능
best_topk = results_df.loc[results_df["RMSE"].idxmin()]

comparison_df = pd.DataFrame([
    {
        "model": "Basic Item-CF",
        "RMSE": rmse_basic
    },
    {
        "model": "Mean-Centered Item-CF",
        "RMSE": rmse_centered
    },
    {
        "model": best_topk["model"],
        "RMSE": best_topk["RMSE"]
    }
])

comparison_df

,model,RMSE
0,Basic Item-CF,1.010580
1,Mean-Centered Item-CF,0.947380
2,Mean-Centered Item-CF Top-30,0.930488


validation 예측

In [56]:
best_k = 30

topk_preds = []

for row in val.itertuples():
    pred = predict_item_based_centered_topk(
        row.user_id,
        row.movie_id,
        train_matrix,
        item_similarity_centered_df,
        user_means,
        K=best_k
    )

    topk_preds.append(pred)

val["item_cf_topk_pred"] = topk_preds

In [57]:
val[[
    "rating",
    "item_cf_topk_pred"
]].head()

,rating,item_cf_topk_pred
30016,5,4.200658
51012,3,3.144088
79374,4,3.651598
30415,5,4.601417
69284,3,3.205588


# SVD

In [2]:
pip install scikit-surprise

In [1]:
!pip install -q numpy==1.26.4 scikit-surprise==1.1.4

In [2]:
# !pip uninstall -y numpy scikit-surprise
# !pip install "numpy<2"
# !pip install scikit-surprise

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scikit-surprise 1.1.4
Uninstalling scikit-surprise-1.1.4:
  Successfully uninstalled scikit-surprise-1.1.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompat

  Using cached scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl


In [3]:
from surprise import Dataset
from surprise import Reader

In [4]:
reader = Reader(rating_scale=(1, 5))

train_inner 변환

In [29]:
train_data = Dataset.load_from_df(
    train_inner[["user_id", "movie_id", "rating"]],
    reader
)

Surprise용 trainset 생성

In [30]:
trainset = train_data.build_full_trainset()

SVD 모델 생성

In [31]:
from surprise import SVD

from surprise import SVD

svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

학습

In [32]:
svd_model.fit(trainset)

validtion 예측

In [33]:
svd_preds = []

for row in val.itertuples():
    pred = svd_model.predict(
        row.user_id,
        row.movie_id
    ).est

    svd_preds.append(pred)

val["svd_pred"] = svd_preds

RMSE 계산

In [34]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse_svd = np.sqrt(
    mean_squared_error(
        val["rating"],
        val["svd_pred"]
    )
)

print("SVD RMSE:", rmse_svd)

SVD RMSE: 0.9319892340008203


## SVD tuning

In [35]:
from surprise import Dataset, Reader, SVD, NMF
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(
    train_inner[["user_id", "movie_id", "rating"]],
    reader
)

trainset = train_data.build_full_trainset()

공동 평가 함수

In [36]:
def evaluate_surprise_model(model, trainset, val):
    model.fit(trainset)

    preds = []
    for row in val.itertuples():
        pred = model.predict(row.user_id, row.movie_id).est
        preds.append(pred)

    rmse = np.sqrt(
        mean_squared_error(val["rating"], preds)
    )

    return rmse, preds

SVD Hyperparameter Tuning

In [37]:
svd_param_grid = [
    {"n_factors": 20, "n_epochs": 20, "lr_all": 0.005, "reg_all": 0.02},
    {"n_factors": 50, "n_epochs": 20, "lr_all": 0.005, "reg_all": 0.02},
    {"n_factors": 100, "n_epochs": 20, "lr_all": 0.005, "reg_all": 0.02},

    {"n_factors": 50, "n_epochs": 30, "lr_all": 0.005, "reg_all": 0.02},
    {"n_factors": 50, "n_epochs": 40, "lr_all": 0.005, "reg_all": 0.02},

    {"n_factors": 50, "n_epochs": 30, "lr_all": 0.002, "reg_all": 0.02},
    {"n_factors": 50, "n_epochs": 30, "lr_all": 0.007, "reg_all": 0.02},

    {"n_factors": 50, "n_epochs": 30, "lr_all": 0.005, "reg_all": 0.01},
    {"n_factors": 50, "n_epochs": 30, "lr_all": 0.005, "reg_all": 0.05},
]

SVD 예측

In [38]:
svd_tuning_results = []

best_svd_rmse = float("inf")
best_svd_model = None
best_svd_preds = None
best_svd_params = None

for params in svd_param_grid:
    model = SVD(
        n_factors=params["n_factors"],
        n_epochs=params["n_epochs"],
        lr_all=params["lr_all"],
        reg_all=params["reg_all"],
        random_state=42
    )

    rmse, preds = evaluate_surprise_model(model, trainset, val)

    result = params.copy()
    result["RMSE"] = rmse
    svd_tuning_results.append(result)

    if rmse < best_svd_rmse:
        best_svd_rmse = rmse
        best_svd_model = model
        best_svd_preds = preds
        best_svd_params = params

svd_tuning_df = pd.DataFrame(svd_tuning_results)
svd_tuning_df.sort_values("RMSE")

,n_factors,n_epochs,lr_all,reg_all,RMSE
8,50,30,0.005,0.05,0.921774
1,50,20,0.005,0.02,0.931989
0,20,20,0.005,0.02,0.935638
3,50,30,0.005,0.02,0.936393
2,100,20,0.005,0.02,0.937125
5,50,30,0.002,0.02,0.939994
7,50,30,0.005,0.01,0.948944
4,50,40,0.005,0.02,0.951812
6,50,30,0.007,0.02,0.954879


In [39]:
val["tuned_svd_pred"] = best_svd_preds

print("Best SVD Params:", best_svd_params)
print("Best Tuned SVD RMSE:", best_svd_rmse)

Best SVD Params: {'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05}
Best Tuned SVD RMSE: 0.9217742191825126


## SVD + Implicit Feedback

In [40]:
from surprise import Dataset, Reader, SVDpp
from sklearn.metrics import mean_squared_error
import numpy as np

reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(
    train_inner[["user_id", "movie_id", "rating"]],
    reader
)

trainset = train_data.build_full_trainset()

In [41]:
svdpp_model = SVDpp(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svdpp_model.fit(trainset)

validation 예측

In [42]:
svdpp_preds = []

for row in val.itertuples():
    pred = svdpp_model.predict(
        row.user_id,
        row.movie_id
    ).est
    svdpp_preds.append(pred)

val["svdpp_pred"] = svdpp_preds

RMSE 계산

In [43]:
rmse_svdpp = np.sqrt(
    mean_squared_error(val["rating"], val["svdpp_pred"])
)

print("SVD++ RMSE:", rmse_svdpp)

SVD++ RMSE: 0.9172769419789413


In [44]:
print("SVD RMSE:", rmse_svd)
print("SVD++ RMSE:", rmse_svdpp)

SVD RMSE: 0.9319892340008203
SVD++ RMSE: 0.9172769419789413


## NMF

In [45]:
nmf_param_grid = [
    {"n_factors": 15, "n_epochs": 50, "reg_pu": 0.06, "reg_qi": 0.06},
    {"n_factors": 30, "n_epochs": 50, "reg_pu": 0.06, "reg_qi": 0.06},
    {"n_factors": 50, "n_epochs": 50, "reg_pu": 0.06, "reg_qi": 0.06},

    {"n_factors": 30, "n_epochs": 100, "reg_pu": 0.06, "reg_qi": 0.06},
    {"n_factors": 30, "n_epochs": 50, "reg_pu": 0.03, "reg_qi": 0.03},
    {"n_factors": 30, "n_epochs": 50, "reg_pu": 0.10, "reg_qi": 0.10},
]

예측

In [46]:
nmf_results = []

best_nmf_rmse = float("inf")
best_nmf_model = None
best_nmf_preds = None
best_nmf_params = None

for params in nmf_param_grid:
    model = NMF(
        n_factors=params["n_factors"],
        n_epochs=params["n_epochs"],
        reg_pu=params["reg_pu"],
        reg_qi=params["reg_qi"],
        random_state=42
    )

    rmse, preds = evaluate_surprise_model(model, trainset, val)

    result = params.copy()
    result["RMSE"] = rmse
    nmf_results.append(result)

    if rmse < best_nmf_rmse:
        best_nmf_rmse = rmse
        best_nmf_model = model
        best_nmf_preds = preds
        best_nmf_params = params

nmf_results_df = pd.DataFrame(nmf_results)
nmf_results_df.sort_values("RMSE")

,n_factors,n_epochs,reg_pu,reg_qi,RMSE
5,30,50,0.10,0.10,0.943913
3,30,100,0.06,0.06,0.952044
0,15,50,0.06,0.06,0.958504
1,30,50,0.06,0.06,0.990478
2,50,50,0.06,0.06,1.032322
4,30,50,0.03,0.03,1.263780


In [47]:
val["nmf_pred"] = best_nmf_preds

print("Best NMF Params:", best_nmf_params)
print("Best NMF RMSE:", best_nmf_rmse)

Best NMF Params: {'n_factors': 30, 'n_epochs': 50, 'reg_pu': 0.1, 'reg_qi': 0.1}
Best NMF RMSE: 0.9439130234149303


모델 기반 CF 전체 비교

In [48]:
model_based_comparison = pd.DataFrame([
    {
        "model": "SVD 기본",
        "RMSE": rmse_svd
    },
    {
        "model": "Tuned SVD",
        "RMSE": best_svd_rmse
    },
    {
        "model": "SVD++",
        "RMSE": rmse_svdpp
    },
    {
        "model": "NMF",
        "RMSE": best_nmf_rmse
    }
])

model_based_comparison.sort_values("RMSE")

,model,RMSE
2,SVD++,0.917277
1,Tuned SVD,0.921774
0,SVD 기본,0.931989
3,NMF,0.943913


# 앙상블
mean-centered + top-k , svd++

In [55]:
print(val.columns)

Index(['user_id', 'movie_id', 'rating', 'item_cf_pred',
       'item_cf_centered_pred', 'svd_pred', 'tuned_svd_pred', 'svdpp_pred',
       'nmf_pred'],
      dtype='object')


In [58]:
val["item_cf_topk_pred"]   # mean-centered + top-k 예측값
val["svdpp_pred"]          # SVD++ 예측값

,svdpp_pred
30016,4.099559
51012,2.988884
79374,3.410853
30415,4.460032
69284,3.662704
...,...
63088,3.663117
34210,4.309503
31141,3.910615
35972,3.195599


## weighted hybrids

가중치 탐색

In [59]:
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

ensemble_results = []

for alpha in np.arange(0, 1.01, 0.05):
    val["ensemble_pred"] = (
        alpha * val["item_cf_topk_pred"] +
        (1 - alpha) * val["svdpp_pred"]
    )

    rmse = np.sqrt(
        mean_squared_error(val["rating"], val["ensemble_pred"])
    )

    ensemble_results.append({
        "alpha_item_cf": alpha,
        "alpha_svdpp": 1 - alpha,
        "RMSE": rmse
    })

ensemble_results_df = pd.DataFrame(ensemble_results)
ensemble_results_df.sort_values("RMSE").head()

,alpha_item_cf,alpha_svdpp,RMSE
8,0.40,0.60,0.908617
7,0.35,0.65,0.908678
9,0.45,0.55,0.908849
6,0.30,0.70,0.909033
10,0.50,0.50,0.909374


최적 가중치 저장

In [60]:
best_ensemble = ensemble_results_df.loc[
    ensemble_results_df["RMSE"].idxmin()
]

best_alpha = best_ensemble["alpha_item_cf"]

print("Best alpha for Item-CF:", best_alpha)
print("Best alpha for SVD++:", 1 - best_alpha)
print("Best Ensemble Validation RMSE:", best_ensemble["RMSE"])

Best alpha for Item-CF: 0.4
Best alpha for SVD++: 0.6
Best Ensemble Validation RMSE: 0.9086169939632358


기존 모델들과 비교

In [61]:
comparison_df = pd.DataFrame([
    {
        "model": "Mean-Centered + Top-K Item-CF",
        "RMSE": np.sqrt(mean_squared_error(val["rating"], val["item_cf_topk_pred"]))
    },
    {
        "model": "SVD++",
        "RMSE": np.sqrt(mean_squared_error(val["rating"], val["svdpp_pred"]))
    },
    {
        "model": "Ensemble",
        "RMSE": best_ensemble["RMSE"]
    }
])

comparison_df.sort_values("RMSE")

,model,RMSE
2,Ensemble,0.908617
1,SVD++,0.917277
0,Mean-Centered + Top-K Item-CF,0.930488


## bagging

In [62]:
from surprise import Dataset, Reader, SVDpp
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

reader = Reader(rating_scale=(1, 5))

In [63]:
def train_svdpp_and_predict(train_df, val_df, seed):
    train_data = Dataset.load_from_df(
        train_df[["user_id", "movie_id", "rating"]],
        reader
    )
    trainset = train_data.build_full_trainset()

    model = SVDpp(
        n_factors=50,
        n_epochs=20,
        lr_all=0.005,
        reg_all=0.02,
        random_state=seed
    )

    model.fit(trainset)

    preds = [
        model.predict(row.user_id, row.movie_id).est
        for row in val_df.itertuples()
    ]

    return preds

SVD++ Bagging

In [64]:
bagging_preds = []

seeds = [0, 1, 2, 3, 4]

for seed in seeds:
    sampled_train = train_inner.sample(
        frac=1.0,
        replace=True,
        random_state=seed
    )

    preds = train_svdpp_and_predict(
        sampled_train,
        val,
        seed
    )

    bagging_preds.append(preds)

val["svdpp_bagging_pred"] = np.mean(bagging_preds, axis=0)

RMSE 확인

In [65]:
rmse_svdpp_bagging = np.sqrt(
    mean_squared_error(
        val["rating"],
        val["svdpp_bagging_pred"]
    )
)

print("SVD++ RMSE:", rmse_svdpp)
print("SVD++ Bagging RMSE:", rmse_svdpp_bagging)

SVD++ RMSE: 0.9172769419789413
SVD++ Bagging RMSE: 0.9179527086065083


가중 하이브리드 vs 배깅

In [66]:
ensemble_comparison = pd.DataFrame([
    {
        "model": "Mean-Centered + Top-K Item-CF",
        "RMSE": np.sqrt(mean_squared_error(val["rating"], val["item_cf_topk_pred"]))
    },
    {
        "model": "SVD++",
        "RMSE": rmse_svdpp
    },
    {
        "model": "SVD++ Bagging",
        "RMSE": rmse_svdpp_bagging
    },
    {
        "model": "Weighted Hybrid(Item-CF + SVD++)",
        "RMSE": best_ensemble["RMSE"]
    }
])

ensemble_comparison.sort_values("RMSE")

,model,RMSE
3,Weighted Hybrid(Item-CF + SVD++),0.908617
1,SVD++,0.917277
2,SVD++ Bagging,0.917953
0,Mean-Centered + Top-K Item-CF,0.930488


Bagging SVD++를 hybrid에 넣기

In [67]:
bagging_hybrid_results = []

for alpha in np.arange(0, 1.01, 0.05):
    pred = (
        alpha * val["item_cf_topk_pred"] +
        (1 - alpha) * val["svdpp_bagging_pred"]
    )

    rmse = np.sqrt(
        mean_squared_error(val["rating"], pred)
    )

    bagging_hybrid_results.append({
        "alpha_item_cf": alpha,
        "alpha_svdpp_bagging": 1 - alpha,
        "RMSE": rmse
    })

bagging_hybrid_df = pd.DataFrame(bagging_hybrid_results)
bagging_hybrid_df.sort_values("RMSE").head()

,alpha_item_cf,alpha_svdpp_bagging,RMSE
7,0.35,0.65,0.910924
8,0.40,0.60,0.910928
6,0.30,0.70,0.911173
9,0.45,0.55,0.911184
5,0.25,0.75,0.911675


In [68]:
best_bagging_hybrid = bagging_hybrid_df.loc[
    bagging_hybrid_df["RMSE"].idxmin()
]

print("Best alpha for Item-CF:", best_bagging_hybrid["alpha_item_cf"])
print("Best alpha for SVD++ Bagging:", best_bagging_hybrid["alpha_svdpp_bagging"])
print("Best Bagging Hybrid RMSE:", best_bagging_hybrid["RMSE"])

Best alpha for Item-CF: 0.35000000000000003
Best alpha for SVD++ Bagging: 0.6499999999999999
Best Bagging Hybrid RMSE: 0.9109242483182436


최종 비교

In [69]:
final_ensemble_comparison = pd.DataFrame([
    {
        "model": "SVD++",
        "RMSE": rmse_svdpp
    },
    {
        "model": "SVD++ Bagging",
        "RMSE": rmse_svdpp_bagging
    },
    {
        "model": "Weighted Hybrid(Item-CF + SVD++)",
        "RMSE": best_ensemble["RMSE"]
    },
    {
        "model": "Weighted Hybrid(Item-CF + SVD++ Bagging)",
        "RMSE": best_bagging_hybrid["RMSE"]
    }
])

final_ensemble_comparison.sort_values("RMSE")

,model,RMSE
2,Weighted Hybrid(Item-CF + SVD++),0.908617
3,Weighted Hybrid(Item-CF + SVD++ Bagging),0.910924
0,SVD++,0.917277
1,SVD++ Bagging,0.917953


## Cascade Hybrids

In [70]:
val["item_cf_topk_pred"]

,item_cf_topk_pred
30016,4.200658
51012,3.144088
79374,3.651598
30415,4.601417
69284,3.205588
...,...
63088,3.435952
34210,4.251776
31141,4.066296
35972,3.004831


사용자별 Top-50을 후보로 남기기

In [95]:
candidate_df = (
    val
    .sort_values(
        ["user_id", "item_cf_topk_pred"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(50)
)

SVD++로 reranking

In [96]:
candidate_df = candidate_df.sort_values(
    ["user_id", "svdpp_pred"],
    ascending=[True, False]
)

최종 추천

In [78]:
final_recommendation = (
    candidate_df
    .groupby("user_id")
    .head(10)
)

final_recommendation.head(10)

,user_id,movie_id,rating,item_cf_pred,item_cf_centered_pred,svd_pred,tuned_svd_pred,svdpp_pred,nmf_pred,item_cf_topk_pred,ensemble_pred,svdpp_bagging_pred
261,1,272,3,3.816110,4.383284,4.630568,4.693052,4.903220,4.602329,4.546442,4.546442,4.951930
179,1,187,4,3.861167,4.353989,4.411241,4.468595,4.714749,4.314769,4.469531,4.469531,4.593250
241,1,251,4,3.937575,4.232872,4.467113,4.522423,4.602067,4.422754,4.405402,4.405402,4.586553
85,1,89,5,3.829832,4.418109,4.381809,4.421581,4.597802,4.353337,4.682132,4.682132,4.382064
53,1,56,4,3.829813,4.303980,4.081642,4.227097,4.513875,4.205330,4.165461,4.165461,4.539143
190,1,199,4,3.865540,4.426256,4.115644,4.217935,4.439107,4.287208,4.587932,4.587932,4.595265
224,1,234,4,3.797558,4.207498,4.207524,4.239269,4.384242,3.999346,4.335096,4.335096,4.545564
54,1,57,5,3.902975,3.845611,4.228845,4.181966,4.373480,3.935657,3.852193,3.852193,4.023373
57,1,60,5,3.942989,4.070643,4.301134,4.354093,4.338249,3.929681,4.141371,4.141371,4.293541
195,1,205,3,3.830262,4.128210,4.025946,4.044732,4.335437,4.171389,4.390525,4.390525,4.073489


특정 사용자 추천 보기

In [79]:
final_recommendation[
    final_recommendation["user_id"] == 1
]

,user_id,movie_id,rating,item_cf_pred,item_cf_centered_pred,svd_pred,tuned_svd_pred,svdpp_pred,nmf_pred,item_cf_topk_pred,ensemble_pred,svdpp_bagging_pred
261,1,272,3,3.816110,4.383284,4.630568,4.693052,4.903220,4.602329,4.546442,4.546442,4.951930
179,1,187,4,3.861167,4.353989,4.411241,4.468595,4.714749,4.314769,4.469531,4.469531,4.593250
241,1,251,4,3.937575,4.232872,4.467113,4.522423,4.602067,4.422754,4.405402,4.405402,4.586553
85,1,89,5,3.829832,4.418109,4.381809,4.421581,4.597802,4.353337,4.682132,4.682132,4.382064
53,1,56,4,3.829813,4.303980,4.081642,4.227097,4.513875,4.205330,4.165461,4.165461,4.539143
190,1,199,4,3.865540,4.426256,4.115644,4.217935,4.439107,4.287208,4.587932,4.587932,4.595265
224,1,234,4,3.797558,4.207498,4.207524,4.239269,4.384242,3.999346,4.335096,4.335096,4.545564
54,1,57,5,3.902975,3.845611,4.228845,4.181966,4.373480,3.935657,3.852193,3.852193,4.023373
57,1,60,5,3.942989,4.070643,4.301134,4.354093,4.338249,3.929681,4.141371,4.141371,4.293541
195,1,205,3,3.830262,4.128210,4.025946,4.044732,4.335437,4.171389,4.390525,4.390525,4.073489


# 평가

test 예측값 출력

In [81]:
best_k = 30

test_item_cf_preds = []

for row in test.itertuples():
    pred = predict_item_based_centered_topk(
        row.user_id,
        row.movie_id,
        train_matrix,
        item_similarity_centered_df,
        user_means,
        K=best_k
    )
    test_item_cf_preds.append(pred)

test["item_cf_topk_pred"] = test_item_cf_preds

In [82]:
test_svdpp_preds = []

for row in test.itertuples():
    pred = svdpp_model.predict(
        row.user_id,
        row.movie_id
    ).est
    test_svdpp_preds.append(pred)

test["svdpp_pred"] = test_svdpp_preds

In [107]:
test_bagging_preds = []

seeds = [0, 1, 2, 3, 4]

for seed in seeds:
    sampled_train = train.sample(
        frac=1.0,
        replace=True,
        random_state=seed
    )

    preds = train_svdpp_and_predict(
        sampled_train,
        test,
        seed
    )

    test_bagging_preds.append(preds)

test["svdpp_bagging_pred"] = np.mean(test_bagging_preds, axis=0)

## weighted hybrids test

In [84]:
test["weighted_hybrid_pred"] = (
    best_alpha * test["item_cf_topk_pred"] +
    (1 - best_alpha) * test["svdpp_pred"]
)

RMSE, MAE 계산

In [85]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse_weighted = np.sqrt(
    mean_squared_error(test["rating"], test["weighted_hybrid_pred"])
)

mae_weighted = mean_absolute_error(
    test["rating"], test["weighted_hybrid_pred"]
)

print("Weighted Hybrid Test RMSE:", rmse_weighted)
print("Weighted Hybrid Test MAE:", mae_weighted)

Weighted Hybrid Test RMSE: 0.9437868823336703
Weighted Hybrid Test MAE: 0.7451385282189719


positive 정의

In [108]:
test["positive"] = (
    test["rating"] >= 4
).astype(int)

Precision@K

In [109]:
import numpy as np

def precision_at_k(df, score_col, k=10):
    precisions = []

    for user_id, group in df.groupby("user_id"):

        top_k = (
            group
            .sort_values(score_col, ascending=False)
            .head(k)
        )

        if len(top_k) == 0:
            continue

        precision = top_k["positive"].sum() / k
        precisions.append(precision)

    return np.mean(precisions)

Recall@K

In [110]:
def recall_at_k(df, score_col, k=10):
    recalls = []

    for user_id, group in df.groupby("user_id"):

        total_positive = group["positive"].sum()

        if total_positive == 0:
            continue

        top_k = (
            group
            .sort_values(score_col, ascending=False)
            .head(k)
        )

        recall = (
            top_k["positive"].sum()
            / total_positive
        )

        recalls.append(recall)

    return np.mean(recalls)

평가

In [111]:
weighted_precision_10 = precision_at_k(
    test,
    score_col="weighted_hybrid_pred",
    k=10
)

weighted_recall_10 = recall_at_k(
    test,
    score_col="weighted_hybrid_pred",
    k=10
)

print("Weighted Hybrid Precision@10:", weighted_precision_10)
print("Weighted Hybrid Recall@10:", weighted_recall_10)

Weighted Hybrid Precision@10: 0.5799575821845174
Weighted Hybrid Recall@10: 1.0


## bagging test

RMSE, MAE 계산

In [112]:
rmse_bagging = np.sqrt(
    mean_squared_error(
        test["rating"],
        test["svdpp_bagging_pred"]
    )
)

mae_bagging = mean_absolute_error(
    test["rating"],
    test["svdpp_bagging_pred"]
)

print("SVD++ Bagging Test RMSE:", rmse_bagging)
print("SVD++ Bagging Test MAE:", mae_bagging)

SVD++ Bagging Test RMSE: 0.941217547329916
SVD++ Bagging Test MAE: 0.7458108851576135


In [113]:
bagging_precision_10 = precision_at_k(
    test,
    score_col="svdpp_bagging_pred",
    k=10
)

bagging_recall_10 = recall_at_k(
    test,
    score_col="svdpp_bagging_pred",
    k=10
)

print("Bagging Precision@10:", bagging_precision_10)
print("Bagging Recall@10:", bagging_recall_10)

Bagging Precision@10: 0.5799575821845174
Bagging Recall@10: 1.0


## Cascade Hybrid test

positive 기준 설정

In [97]:
test["positive"] = (test["rating"] >= 4).astype(int)

In [98]:
candidate_test_df = (
    test
    .sort_values(
        ["user_id", "item_cf_topk_pred"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(50)
)

In [100]:
cascade_test_result = (
    candidate_test_df
    .sort_values(
        ["user_id", "svdpp_pred"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(10)
)

cascade_test_result.head()

,user_id,movie_id,rating,item_cf_topk_pred,svdpp_pred,weighted_hybrid_pred,positive
6,1,171,5,4.501007,4.391036,4.435024,1
7,1,189,3,4.379524,4.330921,4.350362,0
2,1,61,4,4.107382,4.156002,4.136554,1
9,1,265,4,4.386086,4.027164,4.170733,1
8,1,202,5,4.512931,3.753067,4.057013,1


Precision@K

In [87]:
def precision_at_k(df, score_col, k=10):
    precisions = []

    for user_id, group in df.groupby("user_id"):
        top_k = group.sort_values(score_col, ascending=False).head(k)

        if len(top_k) == 0:
            continue

        precisions.append(top_k["positive"].sum() / k)

    return np.mean(precisions)

Recall@K

In [88]:
def recall_at_k(df, score_col, k=10):
    recalls = []

    for user_id, group in df.groupby("user_id"):
        positives = group["positive"].sum()

        if positives == 0:
            continue

        top_k = group.sort_values(score_col, ascending=False).head(k)
        recalls.append(top_k["positive"].sum() / positives)

    return np.mean(recalls)

평가

In [101]:
precision_10 = precision_at_k(
    cascade_test_result,
    score_col="svdpp_pred",
    k=10
)

recall_10 = recall_at_k(
    cascade_test_result,
    score_col="svdpp_pred",
    k=10
)

print("Cascade Precision@10:", precision_10)
print("Cascade Recall@10:", recall_10)

Cascade Precision@10: 0.5799575821845174
Cascade Recall@10: 1.0


In [103]:
cascade_test_result.head(10)

,user_id,movie_id,rating,item_cf_topk_pred,svdpp_pred,weighted_hybrid_pred,positive
6,1,171,5,4.501007,4.391036,4.435024,1
7,1,189,3,4.379524,4.330921,4.350362,0
2,1,61,4,4.107382,4.156002,4.136554,1
9,1,265,4,4.386086,4.027164,4.170733,1
8,1,202,5,4.512931,3.753067,4.057013,1
3,1,117,3,4.002169,3.505137,3.703950,0
1,1,33,4,3.668583,3.337359,3.469848,1
5,1,160,4,3.563839,3.284423,3.396189,1
0,1,20,4,3.691716,3.160994,3.373283,1
4,1,155,2,2.806097,2.877199,2.848758,0


## 평점 예측 성능 비교표

In [114]:
rating_result_df = pd.DataFrame([
    {
        "Model": "Weighted Hybrid",
        "RMSE": rmse_weighted,
        "MAE": mae_weighted
    },
    {
        "Model": "SVD++ Bagging",
        "RMSE": rmse_bagging,
        "MAE": mae_bagging
    }
])

rating_result_df.sort_values("RMSE")

,Model,RMSE,MAE
1,SVD++ Bagging,0.941218,0.745811
0,Weighted Hybrid,0.943787,0.745139


## 추천 품질 비교표

In [115]:
ranking_result_df = pd.DataFrame([
    {
        "Model": "Weighted Hybrid",
        "Precision@10": weighted_precision_10,
        "Recall@10": weighted_recall_10
    },
    {
        "Model": "SVD++ Bagging",
        "Precision@10": bagging_precision_10,
        "Recall@10": bagging_recall_10
    },
    {
        "Model": "Cascade Hybrid",
        "Precision@10": precision_10,
        "Recall@10": recall_10
    }
])

ranking_result_df.sort_values(
    "Precision@10",
    ascending=False
)

,Model,Precision@10,Recall@10
0,Weighted Hybrid,0.579958,1.0
1,SVD++ Bagging,0.579958,1.0
2,Cascade Hybrid,0.579958,1.0


최종 출력

In [116]:
print("===== Rating Prediction Evaluation =====")
display(rating_result_df)

print("\n===== Ranking Evaluation =====")
display(ranking_result_df)

===== Rating Prediction Evaluation =====


,Model,RMSE,MAE
0,Weighted Hybrid,0.943787,0.745139
1,SVD++ Bagging,0.941218,0.745811



===== Ranking Evaluation =====


,Model,Precision@10,Recall@10
0,Weighted Hybrid,0.579958,1.0
1,SVD++ Bagging,0.579958,1.0
2,Cascade Hybrid,0.579958,1.0
